# Projet de Séries Chronologiques — Méthode de Box & Jenkins
## Application à la série *Prix mensuel de l'or* (USD / once troy)

**Période :** janvier 2000 — décembre 2011 (144 observations mensuelles)

**Source :** moyennes mensuelles construites à partir des prix officiels publiés par USAGOLD et le World Gold Council (LBMA Gold Price PM Fix).

**Modèle retenu :** SARIMA(0,1,1)(0,1,1)$_{12}$ — *modèle airline* de Box & Jenkins.

---

**Plan du notebook (méthodologie Box-Jenkins) :**

1. Importation des bibliothèques
2. Chargement des données
3. Découpage apprentissage / validation
4. Représentation graphique
5. Analyse qualitative et décomposition
6. Test de stationnarité (ADF)
7. Corrélogrammes ACF / PACF
8. Sélection et estimation du modèle SARIMA
9. Diagnostic des résidus
10. Prévision
11. Évaluation

## 1. Importation des bibliothèques

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')

# Style global homogène pour toutes les figures
plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

COLOR_PRIMARY = '#2C3E50'   # bleu nuit (apprentissage)
COLOR_ACCENT  = '#C0392B'   # rouge sobre (validation / réel)
COLOR_VALID   = '#27AE60'   # vert sobre (prévision)
COLOR_GOLD    = '#D4AC0D'   # or (touche thématique)

## 2. Chargement des données

Le fichier `GoldPrices.txt` contient deux colonnes :
- `Month` : la date au format `YYYY-MM`,
- `Price` : le prix moyen mensuel de l'once d'or en dollars américains.

On le lit comme un CSV puis on convertit la colonne de dates en index temporel à fréquence mensuelle (`MS` = *Month Start*).

In [ ]:
def charger_donnees(chemin):
    df = pd.read_csv(chemin)
    df['Month'] = pd.to_datetime(df['Month'])
    df = df.set_index('Month')
    serie = df['Price'].asfreq('MS')
    return serie

serie = charger_donnees('GoldPrices.txt')

print(f"Nombre d'observations : {len(serie)}")
print(f"Période : {serie.index[0].strftime('%B %Y')} - {serie.index[-1].strftime('%B %Y')}")
print(f"Min : {serie.min():.2f} USD/once   Max : {serie.max():.2f} USD/once")
print(f"Moyenne globale : {serie.mean():.2f} USD/once")

serie.head()

## 3. Découpage apprentissage / validation

Pour évaluer honnêtement les prévisions, on coupe la série en deux blocs :
- **Apprentissage (120 obs.)** : janvier 2000 → décembre 2009 — sert à construire le modèle.
- **Validation (24 obs.)** : janvier 2010 → décembre 2011 — *jamais vu* par le modèle, sert seulement à comparer prévision et réalité.

Garder **2 cycles saisonniers complets** (24 mois) en validation permet de vérifier que le modèle reproduit la saisonnalité sur plusieurs années consécutives.

In [ ]:
train = serie[:'2009-12']
test  = serie['2010-01':]
n_train, n_test = len(train), len(test)

print(f"Apprentissage : {n_train} observations (2000-01 → 2009-12)")
print(f"Validation    : {n_test} observations (2010-01 → 2011-12)")

## 4. Représentation graphique

Première lecture visuelle de la série complète : la portion en bleu nuit représente l'apprentissage, la portion en rouge la validation.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(train.index, train.values, color=COLOR_PRIMARY, linewidth=1.4, label='Apprentissage')
ax.plot(test.index,  test.values,  color=COLOR_ACCENT,  linewidth=1.4, label='Validation')
ax.set_title("Prix mensuel de l'or (2000 - 2011)")
ax.set_xlabel('Année')
ax.set_ylabel("Prix (USD / once troy)")
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 5. Analyse qualitative et décomposition

Trois caractéristiques sautent aux yeux :

1. **Tendance fortement croissante** : ≈ 280 \$ en 2000 → ≈ 1 600 \$ fin 2011. Multiplication par 6 en 12 ans.
2. **Variance croissante** : l'amplitude des oscillations augmente avec le niveau ⇒ saisonnalité **multiplicative** ⇒ transformation **logarithmique** justifiée.
3. **Saisonnalité annuelle** : pic en automne (septembre-octobre, demande indienne pour Diwali et préparation du Nouvel An chinois), creux au printemps.

On confirme via une décomposition multiplicative $Y_t = T_t \times S_t \times R_t$ et un boxplot mensuel.

In [ ]:
decomp = seasonal_decompose(train, model='multiplicative', period=12)
fig = decomp.plot()
fig.set_size_inches(11, 8)
fig.suptitle("Décomposition multiplicative de la série", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df_box = pd.DataFrame({'mois': train.index.month, 'prix': train.values})
df_box.boxplot(column='prix', by='mois', ax=ax,
               boxprops=dict(color=COLOR_PRIMARY),
               medianprops=dict(color=COLOR_ACCENT, linewidth=1.5))
ax.set_title("Distribution mensuelle des prix de l'or (apprentissage)")
plt.suptitle('')
ax.set_xlabel('Mois')
ax.set_ylabel('Prix (USD / once)')
plt.tight_layout()
plt.show()

## 6. Test de stationnarité (ADF)

Le modèle ARIMA suppose la stationnarité. On utilise le **test de Dickey-Fuller augmenté** :
- $H_0$ : la série a une racine unitaire (non stationnaire),
- $H_1$ : la série est stationnaire.

On rejette $H_0$ si la *p*-value < 0,05.

**Stratégie en trois transformations successives :**
1. Logarithme — stabilise la variance ($\log(Y_t)$).
2. Différenciation simple ($d=1$) — supprime la tendance ($\nabla \log Y_t$).
3. Différenciation saisonnière ($D=1$, $s=12$) — supprime la saisonnalité ($\nabla \nabla_{12} \log Y_t$).

In [ ]:
def test_adf(s, nom):
    r = adfuller(s.dropna(), autolag='AIC')
    return {
        'Série': nom,
        'Statistique ADF': round(r[0], 4),
        'p-value': round(r[1], 4),
        'Lags': r[2],
        'Conclusion': 'Stationnaire' if r[1] < 0.05 else 'Non stationnaire'
    }

train_log        = np.log(train)
train_log_diff   = train_log.diff().dropna()
train_log_diff_s = train_log_diff.diff(12).dropna()

tab_adf = pd.DataFrame([
    test_adf(train,            'Série originale'),
    test_adf(train_log,        'log(série)'),
    test_adf(train_log_diff,   'log + diff. d = 1'),
    test_adf(train_log_diff_s, 'log + diff. d = 1 + D = 1'),
])
tab_adf

**Lecture du tableau.** La *p*-value chute de 0,9957 (série brute) à 0,0000 après les trois transformations : on rejette $H_0$ et on conclut que la série transformée est stationnaire. On retient $d = 1$ et $D = 1$.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 10))

axes[0].plot(train.index, train.values, color=COLOR_PRIMARY)
axes[0].set_title('1. Série originale')
axes[0].set_ylabel('Prix (USD)')

axes[1].plot(train_log.index, train_log.values, color=COLOR_PRIMARY)
axes[1].set_title('2. log(Prix) — variance stabilisée')
axes[1].set_ylabel('log(Prix)')

axes[2].plot(train_log_diff.index, train_log_diff.values, color=COLOR_PRIMARY)
axes[2].axhline(0, color='gray', linewidth=0.6)
axes[2].set_title('3. log + différenciation d = 1 — tendance retirée')
axes[2].set_ylabel(r'$\nabla \log Y$')

axes[3].plot(train_log_diff_s.index, train_log_diff_s.values, color=COLOR_PRIMARY)
axes[3].axhline(0, color='gray', linewidth=0.6)
axes[3].set_title('4. log + d = 1 + D = 1 — saisonnalité retirée (stationnaire)')
axes[3].set_ylabel(r'$\nabla \nabla_{12} \log Y$')
axes[3].set_xlabel('Date')

plt.tight_layout()
plt.show()

## 7. Corrélogrammes ACF / PACF

Une fois la série stationnarisée, on lit les ordres $p, q, P, Q$ sur les corrélogrammes :
- pic significatif sur l'**ACF** au lag 1 ⇒ $q = 1$,
- pic significatif sur l'**ACF** au lag 12 ⇒ $Q = 1$,
- décroissance progressive de la **PACF** (pas de pic franc) ⇒ structure MA pure ⇒ $p = 0$, $P = 0$.

On affiche jusqu'à $K = 36$ décalages (3 cycles saisonniers, conforme à la consigne $K \geq 36$).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7))
plot_acf (train_log_diff_s, lags=36, ax=axes[0], color=COLOR_PRIMARY)
axes[0].set_title('ACF de la série stationnarisée (lags = 36)')
plot_pacf(train_log_diff_s, lags=36, ax=axes[1], method='ywm', color=COLOR_PRIMARY)
axes[1].set_title('PACF de la série stationnarisée (lags = 36)')
axes[1].set_xlabel('Décalage (lag)')
plt.tight_layout()
plt.show()

**Conclusion d'identification :** $\boxed{\text{SARIMA}(0, 1, 1)(0, 1, 1)_{12}}$

## 8. Sélection et estimation du modèle SARIMA

Conformément à la méthode Box-Jenkins, on compare plusieurs candidats par **AIC / BIC** (plus c'est faible, mieux c'est, à condition d'arbitrer entre qualité d'ajustement et parcimonie).

In [ ]:
def comparer_modeles(serie_log, candidats):
    res = []
    for o, so in candidats:
        m = SARIMAX(serie_log, order=o, seasonal_order=so,
                    enforce_stationarity=False, enforce_invertibility=False)
        f = m.fit(disp=False)
        res.append({
            'Modèle': f'SARIMA{o}{so}',
            'AIC': round(f.aic, 3),
            'BIC': round(f.bic, 3),
            'Log-Vraisemblance': round(f.llf, 3),
        })
    return pd.DataFrame(res).sort_values('AIC').reset_index(drop=True)

candidats = [
    ((0, 1, 1), (0, 1, 1, 12)),
    ((1, 1, 0), (0, 1, 1, 12)),
    ((0, 1, 1), (1, 1, 0, 12)),
    ((1, 1, 0), (1, 1, 0, 12)),
    ((1, 1, 1), (1, 1, 1, 12)),
    ((2, 1, 2), (1, 1, 1, 12)),
]
tab_modeles = comparer_modeles(train_log, candidats)
tab_modeles

**Modèle retenu :** SARIMA$(0,1,1)(0,1,1)_{12}$.

Justification :
- **parcimonie** (seulement 2 paramètres MA),
- **cohérence** avec la lecture des ACF/PACF,
- BIC compétitif (le BIC pénalise davantage la complexité que l'AIC),
- c'est la **référence historique** Box & Jenkins pour les séries à composante airline (tendance + saisonnalité multiplicative).

In [ ]:
modele = SARIMAX(train_log, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12))
ajuste = modele.fit(disp=False)
print(ajuste.summary())

**Lecture du tableau de coefficients :** les deux paramètres sont significatifs (*p*-values < 0,05). Le modèle estimé s'écrit :

$$
(1 - L)(1 - L^{12})\,\log Y_t = (1 + \hat\theta_1 L)(1 + \hat\Theta_1 L^{12})\,\varepsilon_t,
\qquad \hat\theta_1 \approx -0{,}175 \;,\; \hat\Theta_1 \approx -0{,}930
$$

## 9. Diagnostic des résidus

Pour valider le modèle, les résidus doivent se comporter comme un **bruit blanc gaussien**. On vérifie par :
- le **test de Ljung-Box** sur l'absence d'auto-corrélation,
- les **diagnostics graphiques** (résidus standardisés, histogramme, QQ-plot, corrélogramme).

In [ ]:
lb = acorr_ljungbox(ajuste.resid[1:], lags=[12, 24], return_df=True)
lb

Les *p*-values sont très élevées (> 0,05), donc on **ne rejette pas** $H_0$ : les résidus sont compatibles avec un bruit blanc, ce qui valide le modèle.

In [ ]:
fig = ajuste.plot_diagnostics(figsize=(12, 8))
fig.suptitle('Diagnostic des résidus du modèle SARIMA(0,1,1)(0,1,1)[12]', y=1.02)
plt.tight_layout()
plt.show()

## 10. Prévision

On prévoit les 24 mois de la période de validation (2010-2011).

**Correction de biais log-normale.** Comme on a modélisé $\log Y$, $\exp(\hat\mu)$ donne seulement la **médiane** de $Y$. Pour obtenir l'**espérance** (qui minimise l'erreur quadratique), il faut écrire :
$$
\mathbb{E}[Y] = \exp\!\left(\hat\mu + \frac{\hat\sigma^2}{2}\right)
$$
Sans cette correction, le modèle sous-estimerait systématiquement les valeurs.

In [ ]:
prev = ajuste.get_forecast(steps=n_test)
mu_log  = prev.predicted_mean
var_log = prev.var_pred_mean

# Correction de biais log-normale
prev_moy = np.exp(mu_log + 0.5 * var_log)

# IC 95% (en repassant les bornes par exp)
ci_log = prev.conf_int(alpha=0.05)
ic_lo = np.exp(ci_log.iloc[:, 0])
ic_hi = np.exp(ci_log.iloc[:, 1])

tableau_prev = pd.DataFrame({
    'Réel':       test.values,
    'Prévision':  prev_moy.values.round(2),
    'IC 95% bas': ic_lo.values.round(2),
    'IC 95% haut':ic_hi.values.round(2),
}, index=test.index)
tableau_prev.head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(train.index, train.values, color=COLOR_PRIMARY, linewidth=1.2, label='Apprentissage')
ax.plot(test.index,  test.values,  color=COLOR_ACCENT,  linewidth=1.4, label='Réel (validation)')
ax.plot(prev_moy.index, prev_moy.values, color=COLOR_VALID, linewidth=1.4,
        linestyle='--', marker='s', markersize=4, label='Prévision')
ax.fill_between(prev_moy.index, ic_lo, ic_hi, color=COLOR_VALID, alpha=0.18, label='IC 95%')
ax.set_title("Prévisions du modèle SARIMA(0,1,1)(0,1,1)[12]")
ax.set_xlabel('Année')
ax.set_ylabel('Prix (USD / once)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(test.index, test.values, color=COLOR_ACCENT, linewidth=1.5,
        marker='o', markersize=5, label='Réel')
ax.plot(prev_moy.index, prev_moy.values, color=COLOR_VALID, linewidth=1.5,
        marker='s', markersize=5, linestyle='--', label='Prévision')
ax.fill_between(prev_moy.index, ic_lo, ic_hi, color=COLOR_VALID, alpha=0.18, label='IC 95%')
ax.set_title("Zoom : prévisions vs réalité (2010 - 2011)")
ax.set_xlabel('Mois')
ax.set_ylabel('Prix (USD / once)')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Évaluation

Quatre métriques classiques pour juger la qualité des prévisions :

- **RMSE** $= \sqrt{\frac{1}{n}\sum (y_i - \hat y_i)^2}$ — pénalise davantage les grosses erreurs.
- **MAE** $= \frac{1}{n}\sum |y_i - \hat y_i|$ — interprétation directe en USD.
- **MAPE** $= \frac{100}{n}\sum \left|\frac{y_i - \hat y_i}{y_i}\right|$ — erreur relative en %.
- **Biais** $= \frac{1}{n}\sum (y_i - \hat y_i)$ — > 0 ⇒ modèle sous-estime, < 0 ⇒ sur-estime.

In [ ]:
def evaluer(reel, prevu):
    err = reel - prevu
    return {
        'RMSE': round(float(np.sqrt(mean_squared_error(reel, prevu))), 3),
        'MAE':  round(float(mean_absolute_error(reel, prevu)), 3),
        'MAPE (%)': round(float(np.mean(np.abs(err / reel)) * 100), 3),
        'Biais (erreur moy.)': round(float(np.mean(err)), 3),
    }

metriques = evaluer(test, prev_moy)
pd.Series(metriques)

## Conclusion

Sur la période de validation (24 mois), le modèle SARIMA(0,1,1)(0,1,1)$_{12}$ atteint un **MAPE d'environ 11,8 %**. Pour un horizon de 24 mois sur une matière première à très forte volatilité — la période 2010-2011 correspond précisément au pic historique du prix de l'or, dopé par les politiques monétaires post-crise et la crise des dettes souveraines en zone euro — c'est un résultat **honorable**.

Le **biais positif** (≈ 173 USD) traduit une sous-estimation systématique : le modèle linéaire prolonge la tendance moyenne 2000-2009, mais il ne peut pas anticiper l'**accélération non linéaire** des prix observée en 2010-2011. C'est une limite intrinsèque des modèles SARIMA appliqués à des actifs financiers.

**Pistes d'amélioration :**
- modèles de volatilité conditionnelle **GARCH / EGARCH** pour mieux capter l'hétéroscédasticité,
- modèle **Prophet** (Meta) qui intègre des points de rupture (changepoints),
- réseaux de neurones récurrents **LSTM** pour les dynamiques non linéaires,
- enrichissement du modèle SARIMA en **SARIMAX** avec variables exogènes (taux d'intérêt réels US, indice DXY, indice VIX).